In [1]:
from pathlib import Path
import json
from PIL import Image
from collections import Counter

# Locate the project root
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "02_processed_data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate 02_processed_data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "02_processed_data"

print("Project root:", PROJECT_ROOT)
print("Processed data:", DATA_ROOT)

# Inspect each dataset split
for split in ["training", "validation", "test"]:

    split_dir = DATA_ROOT / split
    images_dir = split_dir / "images"
    annotations_dir = split_dir / "annotations"

    print(f"\n{'=' * 45}")
    print(f" {split.upper()}")
    print(f"{'=' * 45}")

    # Count images
    image_files = list(images_dir.glob("*.jpg"))
    print("Images:", len(image_files))

    # Inspect annotation files
    for filename in [
        "captions.json",
        "instances.json",
        "discarded_captions.json"
    ]:
        file_path = annotations_dir / filename

        if file_path.exists():
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            print(f"\n{filename}")
            print("Top-level keys:", list(data.keys()))

            if "annotations" in data:
                print("Number of annotations:", len(data["annotations"]))
            if "images" in data:
                print("Number of image records:", len(data["images"]))
        else:
            print(f"\n{filename}: NOT FOUND")

Project root: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG
Processed data: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data

 TRAINING
Images: 9900

captions.json
Top-level keys: ['info', 'licenses', 'images', 'annotations']
Number of annotations: 49500
Number of image records: 9900

instances.json
Top-level keys: ['info', 'licenses', 'images', 'annotations', 'categories']
Number of annotations: 71326
Number of image records: 9900

discarded_captions.json
Top-level keys: ['split', 'selection_rule', 'discarded_caption_count', 'discarded_annotations']

 VALIDATION
Images: 100

captions.json
Top-level keys: ['info', 'licenses', 'images', 'annotations']
Number of annotations: 500
Number of image records: 100

instances.json
Top-level keys: ['info', 'licenses', 'images', 'annotations', 'categories']
Number of annotations: 769
Number of image records: 100

discarded_captions.json
Top-level keys: ['split', 'selection_rule', 'discarded_caption_count', 'd

In [2]:
for split in ["training", "validation", "test"]:

    images_dir = DATA_ROOT / split / "images"
    image_files = sorted(images_dir.glob("*.jpg"))

    readable = 0
    corrupted = []

    for image_path in image_files:
        try:
            with Image.open(image_path) as img:
                img.verify()
            readable += 1

        except Exception as error:
            corrupted.append((image_path.name, str(error)))

    print(f"\n{split.upper()}")
    print("Total images:", len(image_files))
    print("Readable images:", readable)
    print("Corrupted images:", len(corrupted))

    if corrupted:
        print("Examples:", corrupted[:5])


TRAINING
Total images: 9900
Readable images: 9900
Corrupted images: 0

VALIDATION
Total images: 100
Readable images: 100
Corrupted images: 0

TEST
Total images: 100
Readable images: 100
Corrupted images: 0


In [3]:
from collections import Counter
from PIL import Image
import statistics

for split in ["training", "validation", "test"]:

    images_dir = DATA_ROOT / split / "images"
    image_files = sorted(images_dir.glob("*.jpg"))

    dimensions = Counter()
    modes = Counter()
    exif_orientation = Counter()

    widths = []
    heights = []

    for image_path in image_files:
        with Image.open(image_path) as img:

            width, height = img.size

            dimensions[(width, height)] += 1
            modes[img.mode] += 1

            widths.append(width)
            heights.append(height)

            orientation = img.getexif().get(274, 1)
            exif_orientation[orientation] += 1

    print(f"\n{'=' * 45}")
    print(split.upper())
    print(f"{'=' * 45}")

    print("Image count:", len(image_files))
    print("Colour modes:", modes)
    print("Most common dimensions:", dimensions.most_common(10))
    print("Width range:", min(widths), "-", max(widths))
    print("Height range:", min(heights), "-", max(heights))
    print("EXIF orientations:", exif_orientation)


TRAINING
Image count: 9900
Colour modes: Counter({'RGB': 9881, 'L': 19})
Most common dimensions: [((640, 480), 2092), ((640, 427), 1185), ((480, 640), 742), ((640, 426), 450), ((500, 375), 396), ((427, 640), 338), ((640, 428), 287), ((640, 425), 274), ((612, 612), 196), ((375, 500), 173)]
Width range: 152 - 640
Height range: 109 - 640
EXIF orientations: Counter({1: 9900})

VALIDATION
Image count: 100
Colour modes: Counter({'RGB': 99, 'L': 1})
Most common dimensions: [((640, 480), 16), ((640, 427), 14), ((480, 640), 5), ((640, 426), 5), ((500, 375), 5), ((640, 428), 4), ((427, 640), 4), ((375, 500), 4), ((640, 478), 3), ((640, 425), 2)]
Width range: 299 - 640
Height range: 238 - 640
EXIF orientations: Counter({1: 100})

TEST
Image count: 100
Colour modes: Counter({'RGB': 100})
Most common dimensions: [((640, 480), 22), ((500, 375), 8), ((640, 427), 7), ((640, 425), 7), ((426, 640), 5), ((640, 428), 4), ((640, 426), 4), ((427, 640), 3), ((480, 640), 3), ((640, 434), 2)]
Width range: 256

In [4]:
#%pip install transformers torch pillow

In [5]:
from transformers import CLIPProcessor
from PIL import Image, ImageOps
import torch

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

print("CLIP processor loaded successfully.")

CLIP processor loaded successfully.


In [6]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "01_src"))

from data.preprocessing.image_preprocessing import (
    load_and_prepare_image,
    preprocess_image_for_clip
)

# Select one training image
sample_image_path = next(
    (DATA_ROOT / "training" / "images").glob("*.jpg")
)

# Load and prepare the image
prepared_image = load_and_prepare_image(sample_image_path)

# Apply the CLIP processor
pixel_values = preprocess_image_for_clip(
    sample_image_path,
    clip_processor
)

print("Original image:", sample_image_path.name)
print("Prepared image mode:", prepared_image.mode)
print("Prepared image dimensions:", prepared_image.size)
print("CLIP tensor shape:", pixel_values.shape)
print("Tensor data type:", pixel_values.dtype)

Original image: 000000000149.jpg
Prepared image mode: RGB
Prepared image dimensions: (640, 428)
CLIP tensor shape: torch.Size([1, 3, 224, 224])
Tensor data type: torch.float32


In [7]:
from collections import Counter
from tqdm.auto import tqdm
import torch

for split in ["training", "validation", "test"]:

    images_dir = DATA_ROOT / split / "images"
    image_files = sorted(images_dir.glob("*.jpg"))

    successful = 0
    failed = []

    for image_path in tqdm(image_files, desc=f"Checking {split}"):

        try:
            pixel_values = preprocess_image_for_clip(
                image_path,
                clip_processor
            )

            assert pixel_values.shape == (1, 3, 224, 224)
            assert pixel_values.dtype == torch.float32
            assert torch.isfinite(pixel_values).all()

            successful += 1

        except Exception as error:
            failed.append((image_path.name, str(error)))

    print(f"\n{split.upper()}")
    print("Total images:", len(image_files))
    print("Successfully preprocessed:", successful)
    print("Failed:", len(failed))

    if failed:
        print("Examples of failures:", failed[:5])

Checking training:   0%|          | 0/9900 [00:00<?, ?it/s]


TRAINING
Total images: 9900
Successfully preprocessed: 9900
Failed: 0


Checking validation:   0%|          | 0/100 [00:00<?, ?it/s]


VALIDATION
Total images: 100
Successfully preprocessed: 100
Failed: 0


Checking test:   0%|          | 0/100 [00:00<?, ?it/s]


TEST
Total images: 100
Successfully preprocessed: 100
Failed: 0


In [8]:
import json

validation_results = {
    "stage": "image_preprocessing",
    "model": CLIP_MODEL_NAME,
    "input_format": "RGB",
    "tensor_shape": [1, 3, 224, 224],
    "tensor_dtype": "float32",
    "results": {
        "training": {
            "total": 9900,
            "successful": 9900,
            "failed": 0
        },
        "validation": {
            "total": 100,
            "successful": 100,
            "failed": 0
        },
        "test": {
            "total": 100,
            "successful": 100,
            "failed": 0
        }
    },
    "status": "passed"
}

output_path = DATA_ROOT / "preprocessing_validation.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(validation_results, f, indent=4)

print("Validation results saved to:", output_path)

Validation results saved to: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\preprocessing_validation.json


In [9]:
import json
from pathlib import Path

# Load the training captions
captions_path = (
    DATA_ROOT / "training" / "annotations" / "captions.json"
)

with open(captions_path, "r", encoding="utf-8") as f:
    captions_data = json.load(f)

# Inspect the structure
print("Top-level keys:", list(captions_data.keys()))

print("\nNumber of image records:",
      len(captions_data["images"]))

print("Number of caption annotations:",
      len(captions_data["annotations"]))

# Inspect one image record
print("\nExample image record:")
print(captions_data["images"][0])

# Inspect three caption annotations
print("\nExample captions:")
for annotation in captions_data["annotations"][:3]:
    print(annotation)

Top-level keys: ['info', 'licenses', 'images', 'annotations']

Number of image records: 9900
Number of caption annotations: 49500

Example image record:
{'license': 3, 'file_name': '000000184613.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000184613.jpg', 'height': 336, 'width': 500, 'date_captured': '2013-11-14 12:36:29', 'flickr_url': 'http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg', 'id': 184613}

Example captions:
{'image_id': 149, 'id': 481005, 'caption': 'People gathered outside in a big field on a cloudy day'}
{'image_id': 149, 'id': 481854, 'caption': 'People are flying kites on a high school field.'}
{'image_id': 149, 'id': 483387, 'caption': 'People flying kites in a field on a cloudy day.'}


In [10]:
# Display 10 original captions
for i, annotation in enumerate(
    captions_data["annotations"][:10], start=1
):
    print(f"{i}. {annotation['caption']!r}")

1. 'People gathered outside in a big field on a cloudy day'
2. 'People are flying kites on a high school field.'
3. 'People flying kites in a field on a cloudy day.'
4. 'A host of kites being flown on a field by people.'
5. 'a bunch of people are at a grass field '
6. 'There is a small bus with several people standing next to it.'
7. 'people standing besides a bus taking to each other'
8. 'People eating from food trucks near a commemorative archway'
9. 'Several people walking on a sidewalk near a large arch with figures on it.'
10. 'A very big pretty arch way with a bunch of people near it.'


In [11]:
captions = [
    annotation["caption"]
    for annotation in captions_data["annotations"]
]

print("Total captions:", len(captions))
print("Empty captions:", sum(not c.strip() for c in captions))
print("Captions with leading/trailing whitespace:",
      sum(c != c.strip() for c in captions))
print("Captions with repeated whitespace:",
      sum(c != " ".join(c.split()) for c in captions))

print("Unique caption strings:", len(set(captions)))

Total captions: 49500
Empty captions: 0
Captions with leading/trailing whitespace: 7072
Captions with repeated whitespace: 7555
Unique caption strings: 48996


In [12]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "01_src"))

from data.preprocessing.text_preprocessing import clean_caption

# Test examples
test_captions = [
    "  A person is standing near a car.  ",
    "People   are   flying kites.",
    "A host of kites being flown on a field by people.",
    "The man is sitting under a tree, with a dog."
]

for caption in test_captions:
    print("Original:", repr(caption))
    print("Cleaned: ", repr(clean_caption(caption)))
    print("-" * 50)

Original: '  A person is standing near a car.  '
Cleaned:  'A person is standing near a car.'
--------------------------------------------------
Original: 'People   are   flying kites.'
Cleaned:  'People are flying kites.'
--------------------------------------------------
Original: 'A host of kites being flown on a field by people.'
Cleaned:  'A host of kites being flown on a field by people.'
--------------------------------------------------
Original: 'The man is sitting under a tree, with a dog.'
Cleaned:  'The man is sitting under a tree, with a dog.'
--------------------------------------------------


In [13]:
cleaned_captions = [
    clean_caption(annotation["caption"])
    for annotation in captions_data["annotations"]
]

print("Original captions:", len(captions))
print("Cleaned captions:", len(cleaned_captions))

print("Empty cleaned captions:",
      sum(not caption for caption in cleaned_captions))

print("Remaining leading/trailing whitespace:",
      sum(caption != caption.strip()
          for caption in cleaned_captions))

print("Remaining repeated whitespace:",
      sum(caption != " ".join(caption.split())
          for caption in cleaned_captions))

print("Unique cleaned captions:", len(set(cleaned_captions)))

Original captions: 49500
Cleaned captions: 49500
Empty cleaned captions: 0
Remaining leading/trailing whitespace: 0
Remaining repeated whitespace: 0
Unique cleaned captions: 48949


In [14]:
import json

for split in ["training", "validation", "test"]:

    annotations_dir = DATA_ROOT / split / "annotations"
    captions_dir = DATA_ROOT / split / "captions"

    # Create captions directory if it doesn't exist
    captions_dir.mkdir(parents=True, exist_ok=True)

    input_path = annotations_dir / "captions.json"
    output_path = captions_dir / "captions_cleaned.json"

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Preserve all annotation fields and metadata
    cleaned_data = data.copy()

    cleaned_data["annotations"] = [
        {
            **annotation,
            "caption": clean_caption(annotation["caption"])
        }
        for annotation in data["annotations"]
    ]

    # Save cleaned captions separately
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

    print(f"\n{split.upper()}")
    print("Original captions:", len(data["annotations"]))
    print("Cleaned captions:", len(cleaned_data["annotations"]))
    print("Saved to:", output_path)


TRAINING
Original captions: 49500
Cleaned captions: 49500
Saved to: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\captions\captions_cleaned.json

VALIDATION
Original captions: 500
Cleaned captions: 500
Saved to: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\captions\captions_cleaned.json

TEST
Original captions: 500
Cleaned captions: 500
Saved to: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\captions\captions_cleaned.json


In [15]:
import json

for split in ["training", "validation", "test"]:

    path = (
        DATA_ROOT / split / "captions" / "captions_cleaned.json"
    )

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    captions = [
        annotation["caption"]
        for annotation in data["annotations"]
    ]

    print(f"\n{split.upper()}")
    print("Caption count:", len(captions))

    print(
        "Empty captions:",
        sum(not caption for caption in captions)
    )

    print(
        "Whitespace issues:",
        sum(caption != " ".join(caption.split())
            for caption in captions)
    )


TRAINING
Caption count: 49500
Empty captions: 0
Whitespace issues: 0

VALIDATION
Caption count: 500
Empty captions: 0
Whitespace issues: 0

TEST
Caption count: 500
Empty captions: 0
Whitespace issues: 0


In [16]:
print("CLIP image processor configuration:")
print(clip_processor.image_processor)

CLIP image processor configuration:
CLIPImageProcessorPil {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "CLIPImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}



In [17]:
import json
from PIL import Image
from collections import Counter
from pathlib import Path
import statistics

for split in ["training", "validation", "test"]:

    split_dir = DATA_ROOT / split
    images_dir = split_dir / "images"
    annotations_dir = split_dir / "annotations"
    captions_dir = split_dir / "captions"
    metadata_dir = split_dir / "metadata"

    # Create metadata directory
    metadata_dir.mkdir(parents=True, exist_ok=True)

    # Load original COCO image records
    with open(annotations_dir / "captions.json", "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    # Load cleaned captions
    with open(captions_dir / "captions_cleaned.json", "r", encoding="utf-8") as f:
        cleaned_data = json.load(f)

    # ----------------------------------
    # 1. IMAGE METADATA
    # ----------------------------------

    image_metadata = []

    for image_record in coco_data["images"]:

        filename = image_record["file_name"]
        image_path = images_dir / filename

        with Image.open(image_path) as img:
            image_metadata.append({
                "image_id": image_record["id"],
                "file_name": filename,
                "width": img.width,
                "height": img.height,
                "mode": img.mode,
                "format": img.format,
                "file_size_bytes": image_path.stat().st_size,
                "exif_orientation": img.getexif().get(274, 1)
            })

    image_metadata_output = {
        "split": split,
        "image_count": len(image_metadata),
        "images": image_metadata
    }

    with open(metadata_dir / "image_metadata.json", "w", encoding="utf-8") as f:
        json.dump(image_metadata_output, f, ensure_ascii=False, indent=2)

    # ----------------------------------
    # 2. CAPTION METADATA
    # ----------------------------------

    captions = [
        annotation["caption"]
        for annotation in cleaned_data["annotations"]
    ]

    word_counts = [len(caption.split()) for caption in captions]

    caption_metadata = {
        "split": split,
        "caption_count": len(captions),
        "unique_caption_count": len(set(captions)),
        "empty_caption_count": sum(not caption for caption in captions),
        "word_statistics": {
            "average": statistics.mean(word_counts) if word_counts else 0,
            "median": statistics.median(word_counts) if word_counts else 0,
            "minimum": min(word_counts) if word_counts else 0,
            "maximum": max(word_counts) if word_counts else 0
        },
        "cleaning_operations": [
            "Normalize consecutive whitespace",
            "Remove leading and trailing whitespace"
        ]
    }

    with open(metadata_dir / "caption_metadata.json", "w", encoding="utf-8") as f:
        json.dump(caption_metadata, f, ensure_ascii=False, indent=2)

    # ----------------------------------
    # 3. PREPROCESSING METADATA
    # ----------------------------------

    preprocessing_metadata = {
        "split": split,
        "image_preprocessing": {
            "exif_orientation_correction": True,
            "colour_conversion": "RGB",
            "image_processor": "CLIPProcessor",
            "model": "openai/clip-vit-base-patch32",
            "output_tensor_shape": [1, 3, 224, 224],
            "output_dtype": "float32",
            "operations": [
                "EXIF orientation correction",
                "RGB conversion",
                "Model-specific resizing",
                "Center cropping",
                "Pixel normalization"
            ]
        },
        "caption_preprocessing": {
            "cleaning_function": "clean_caption",
            "operations": [
                "Whitespace normalization",
                "Leading and trailing whitespace removal"
            ],
            "preserve_original_captions": True,
            "text_embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
        }
    }

    with open(metadata_dir / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
        json.dump(preprocessing_metadata, f, ensure_ascii=False, indent=2)

    print(f"\n{split.upper()}")
    print("Images recorded:", len(image_metadata))
    print("Captions recorded:", len(captions))
    print("Metadata saved in:", metadata_dir)


TRAINING
Images recorded: 9900
Captions recorded: 49500
Metadata saved in: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\metadata

VALIDATION
Images recorded: 100
Captions recorded: 500
Metadata saved in: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\metadata

TEST
Images recorded: 100
Captions recorded: 500
Metadata saved in: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\metadata


In [18]:
for split in ["training", "validation", "test"]:

    metadata_dir = DATA_ROOT / split / "metadata"

    print(f"\n{'=' * 40}")
    print(split.upper())
    print(f"{'=' * 40}")

    for filename in [
        "image_metadata.json",
        "caption_metadata.json",
        "preprocessing_metadata.json"
    ]:

        path = metadata_dir / filename

        print(filename, ":", "EXISTS" if path.exists() else "MISSING")

    # Display caption statistics
    with open(metadata_dir / "caption_metadata.json", "r", encoding="utf-8") as f:
        caption_info = json.load(f)

    print("Caption count:", caption_info["caption_count"])
    print("Unique captions:", caption_info["unique_caption_count"])
    print("Average words:", caption_info["word_statistics"]["average"])


TRAINING
image_metadata.json : EXISTS
caption_metadata.json : EXISTS
preprocessing_metadata.json : EXISTS
Caption count: 49500
Unique captions: 48949
Average words: 10.462121212121213

VALIDATION
image_metadata.json : EXISTS
caption_metadata.json : EXISTS
preprocessing_metadata.json : EXISTS
Caption count: 500
Unique captions: 499
Average words: 10.458

TEST
image_metadata.json : EXISTS
caption_metadata.json : EXISTS
preprocessing_metadata.json : EXISTS
Caption count: 500
Unique captions: 499
Average words: 10.412


In [19]:
#%pip install -U sentence-transformers

In [20]:
import sentence_transformers

print("Sentence-Transformers version:",
      sentence_transformers.__version__)

Sentence-Transformers version: 6.2.0


In [22]:

import json
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import torch
import transformers
import PIL
import sentence_transformers

from transformers import CLIPProcessor
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. Model identifiers
# --------------------------------------------------

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


# --------------------------------------------------
# 2. Load the exact models and configurations
# --------------------------------------------------

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

sbert_model = SentenceTransformer(SBERT_MODEL_NAME)


# --------------------------------------------------
# 3. Record software versions
# --------------------------------------------------

software_versions = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "pillow": PIL.__version__,
    "sentence_transformers": sentence_transformers.__version__,
}


# --------------------------------------------------
# 4. Record CLIP configuration
# --------------------------------------------------

clip_configuration = {
    "model_name": CLIP_MODEL_NAME,
    "processor_configuration": clip_processor.to_dict(),
    "image_processor_configuration": clip_processor.image_processor.to_dict(),
    "tokenizer_configuration": clip_processor.tokenizer.init_kwargs,
}


# --------------------------------------------------
# 5. Record Sentence-BERT configuration
# --------------------------------------------------

sbert_transformer = sbert_model[0]

sbert_configuration = {
    "model_name": SBERT_MODEL_NAME,
    "max_sequence_length": sbert_model.max_seq_length,
    "embedding_dimension": sbert_model.get_sentence_embedding_dimension(),
    "similarity_function": str(sbert_model.similarity_fn_name),
    "transformer_model_configuration":
        sbert_transformer.auto_model.config.to_dict(),
    "tokenizer_configuration":
        sbert_transformer.tokenizer.init_kwargs,
    "modules": [
        {
            "name": name,
            "type": type(module).__name__
        }
        for name, module in sbert_model._modules.items()
    ],
}


# --------------------------------------------------
# 6. Combine reproducibility information
# --------------------------------------------------

reproducibility_metadata = {
    "software_versions": software_versions,
    "models": {
        "clip": clip_configuration,
        "sentence_bert": sbert_configuration,
    },
}


# --------------------------------------------------
# 7. Update existing metadata for all splits
# --------------------------------------------------

for split in ["training", "validation", "test"]:

    metadata_path = (
        DATA_ROOT / split / "metadata" / "preprocessing_metadata.json"
    )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    metadata["reproducibility"] = reproducibility_metadata

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

    print(f"Updated: {metadata_path}")


print("\nSoftware versions:")
print(json.dumps(software_versions, indent=2))

print("\nSentence-BERT embedding dimension:",
      sbert_model.get_sentence_embedding_dimension())

print("\nReproducibility metadata successfully recorded.")


Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'openai/clip-vit-base-patch32'.
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/openai/clip-vit-base-p

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\iso99\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\iso99\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (8 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Updated: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\metadata\preprocessing_metadata.json
Updated: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\metadata\preprocessing_metadata.json
Updated: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\metadata\preprocessing_metadata.json

Software versions:
{
  "python": "3.11.4",
  "pytorch": "2.15.0+cpu",
  "transformers": "5.22.1",
  "pillow": "10.3.0",
  "sentence_transformers": "6.2.0"
}

Sentence-BERT embedding dimension: 384

Reproducibility metadata successfully recorded.


C:\Users\iso99\AppData\Local\Temp\ipykernel_8060\471489766.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "embedding_dimension": sbert_model.get_sentence_embedding_dimension(),
C:\Users\iso99\AppData\Local\Temp\ipykernel_8060\471489766.py:120: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sbert_model.get_sentence_embedding_dimension())


In [23]:

for split in ["training", "validation", "test"]:

    metadata_path = (
        DATA_ROOT / split / "metadata" / "preprocessing_metadata.json"
    )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    reproducibility = metadata["reproducibility"]

    print(f"\n--- {split.upper()} ---")
    print("Python:", reproducibility["software_versions"]["python"])
    print("PyTorch:", reproducibility["software_versions"]["pytorch"])
    print("Transformers:", reproducibility["software_versions"]["transformers"])
    print("Pillow:", reproducibility["software_versions"]["pillow"])
    print("Sentence-Transformers:",
          reproducibility["software_versions"]["sentence_transformers"])

    print("CLIP:", reproducibility["models"]["clip"]["model_name"])
    print("Sentence-BERT:",
          reproducibility["models"]["sentence_bert"]["model_name"])
    print("SBERT dimension:",
          reproducibility["models"]["sentence_bert"]["embedding_dimension"])



--- TRAINING ---
Python: 3.11.4
PyTorch: 2.15.0+cpu
Transformers: 5.22.1
Pillow: 10.3.0
Sentence-Transformers: 6.2.0
CLIP: openai/clip-vit-base-patch32
Sentence-BERT: sentence-transformers/all-MiniLM-L6-v2
SBERT dimension: 384

--- VALIDATION ---
Python: 3.11.4
PyTorch: 2.15.0+cpu
Transformers: 5.22.1
Pillow: 10.3.0
Sentence-Transformers: 6.2.0
CLIP: openai/clip-vit-base-patch32
Sentence-BERT: sentence-transformers/all-MiniLM-L6-v2
SBERT dimension: 384

--- TEST ---
Python: 3.11.4
PyTorch: 2.15.0+cpu
Transformers: 5.22.1
Pillow: 10.3.0
Sentence-Transformers: 6.2.0
CLIP: openai/clip-vit-base-patch32
Sentence-BERT: sentence-transformers/all-MiniLM-L6-v2
SBERT dimension: 384
